# PAVSig (Polish child) WER / error-type analysis

This notebook evaluates models in `inclusive-asr-moe2/final_weights` on the Polish child test set (PAVSig) and breaks down errors by type (substitution / deletion / insertion), by word, and by child (speaker). Configure paths and device below, then run the evaluation cells.

In [165]:
# Minimal imports and paths
import json, re
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt

TEST_MANIFEST = Path('/lp-dev/amelia/data/pavsig/training/new_test_manifest_mono.jsonl')
print('TEST_MANIFEST:', TEST_MANIFEST)


TEST_MANIFEST: /lp-dev/amelia/data/pavsig/training/new_test_manifest_mono.jsonl


In [142]:
def load_manifest(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, 'r', encoding='utf-8') as f:
        records = [json.loads(l) for l in f if l.strip()]
    return records


def speaker_from_path(audio_path: str) -> str:
    parts = Path(audio_path).parts
    if len(parts) >= 3:
        return parts[-3]
    return 'unknown'


def normalize_polish_text(text: str) -> str:
    if not text:
        return ''
    t = text.lower().strip()
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"[^\w'\sąćęłńóżźĄĆĘŁŃÓŻŹ-]", ' ', t, flags=re.U)
    t = re.sub(r"[-]", ' ', t)
    t = re.sub(r"\s+", ' ', t).strip()
    return t

In [143]:
def alignment_counts(ref_words, hyp_words):
    n = len(ref_words)
    m = len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    i, j = n, m
    S = D = I = 0
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:
            ops.append(('equal', ref_words[i - 1], hyp_words[j - 1]))
            i -= 1; j -= 1
        else:
            if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
                S += 1
                ops.append(('sub', ref_words[i - 1], hyp_words[j - 1]))
                i -= 1; j -= 1
            elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
                D += 1
                ops.append(('del', ref_words[i - 1], None))
                i -= 1
            else:
                I += 1
                ops.append(('ins', None, hyp_words[j - 1]))
                j -= 1
    ops.reverse()
    return S, D, I, ops

In [144]:
def analyze_manifest_with_model(manifest_path, model, text_norm_fn=normalize_polish_text, sample_limit=None):
    records = load_manifest(manifest_path)
    if sample_limit:
        records = records[:sample_limit]
    audio_files = [r['audio_filepath'] for r in records]
    refs = [r.get('text','') for r in records]

    utterances = []
    per_speaker = defaultdict(lambda: {'S':0,'D':0,'I':0,'words':0,'utt':0})
    word_del = Counter(); word_sub = Counter(); word_ins = Counter()
    sub_conf = defaultdict(Counter)

    for audio, ref in zip(audio_files, refs):
        # transcribe per-file to avoid batching shape issues
        try:
            raw = model.transcribe([audio], batch_size=1, num_workers=0, verbose=False)
            hyp = raw[0].text if hasattr(raw[0], 'text') else str(raw[0])
        except Exception as e:
            hyp = f"__ERROR__:{e}"
        ref_n = text_norm_fn(ref)
        hyp_n = text_norm_fn(hyp)
        ref_w = ref_n.split()
        hyp_w = hyp_n.split()
        S, D, I, ops = alignment_counts(ref_w, hyp_w)
        sp = speaker_from_path(audio)
        per_speaker[sp]['S'] += S
        per_speaker[sp]['D'] += D
        per_speaker[sp]['I'] += I
        per_speaker[sp]['words'] += len(ref_w)
        per_speaker[sp]['utt'] += 1
        for op, rw, hw in ops:
            if op == 'del' and rw:
                word_del[rw] += 1
            elif op == 'ins' and hw:
                word_ins[hw] += 1
            elif op == 'sub' and rw is not None and hw is not None:
                word_sub[rw] += 1
                sub_conf[rw][hw] += 1
        utterances.append({'audio': audio, 'speaker': sp, 'ref': ref_n, 'hyp': hyp_n, 'S': S, 'D': D, 'I': I})

    speaker_rows = []
    total_S = total_D = total_I = total_words = 0
    for sp, v in per_speaker.items():
        S=v['S']; D=v['D']; I=v['I']; W=v['words']
        wer = (S+D+I)/W if W>0 else float('inf')
        speaker_rows.append({'speaker':sp,'S':S,'D':D,'I':I,'words':W,'utt':v['utt'],'WER':wer})
        total_S+=S; total_D+=D; total_I+=I; total_words+=W

    overall = {'S': total_S, 'D': total_D, 'I': total_I, 'words': total_words, 'WER': (total_S+total_D+total_I)/total_words if total_words>0 else float('inf')}

    return {
        'utterances': utterances,
        'per_speaker': pd.DataFrame(sorted(speaker_rows, key=lambda r: r['WER'], reverse=True)),
        'word_del_counts': word_del,
        'word_sub_counts': word_sub,
        'word_ins_counts': word_ins,
        'substitution_confusions': {k: dict(v) for k,v in sub_conf.items()},
        'summary': overall
    }

In [ ]:
models_to_run = [
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_fastconformer.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_moe.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_fastconformer.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_moe_lb_off.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_moe_lb_on.nemo'),
]

all_results = {}

print('Ready to run evaluation (do not re-run the cell that redefines models).')

Ready to run evaluation (do not re-run the cell that redefines models).


## Run evaluation (execute when ready)

The next cell will load each selected model, run transcription on the PAVSig manifest, and store per-model analysis under `all_results`.

In [157]:
# Evaluation loop: imports of heavy libs happen here to use the notebook kernel's environment
import json
import tarfile
import torch
from nemo.collections.asr.models import EncDecCTCModelBPE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

all_results = {}
out_dir = Path('analysis_outputs')
out_dir.mkdir(exist_ok=True)

for mpath in models_to_run:
    mpath = Path(mpath)
    mname = mpath.name
    print(f"=== Evaluating {mname} on {TEST_MANIFEST} ===")
    print('  model path:', mpath)
    if not mpath.exists():
        print('  missing file, skipping')
        all_results[mname] = {'error': 'file not found'}
        continue
    print('  size:', mpath.stat().st_size)
    # quick archive sanity check
    try:
        tf = tarfile.open(mpath, 'r')
        tf.getmembers()[:1]
        tf.close()
        print('  tar check: OK')
    except Exception as e:
        print('  tar check failed:', e)
    try:
        model = EncDecCTCModelBPE.restore_from(str(mpath), map_location=device)
        model.to(device)
    except Exception as e:
        print('Restore error for', mname, e)
        all_results[mname] = {'error': str(e)}
        (out_dir / f"{mname}_error.json").write_text(json.dumps({'model': mname, 'error': str(e), 'path': str(mpath)}))
        continue
    try:
        res = analyze_manifest_with_model(TEST_MANIFEST, model)
        all_results[mname] = res
        # save brief outputs
        (out_dir / f"{mname}_summary.json").write_text(json.dumps({'model': mname, 'summary': res['summary'], 'path': str(mpath)}))
        res['per_speaker'].to_csv(out_dir / f"{mname}_per_speaker.csv", index=False)
        print('Saved results for', mname)
    except Exception as e:
        print('Eval error for', mname, e)
        all_results[mname] = {'error': str(e)}
        (out_dir / f"{mname}_error.json").write_text(json.dumps({'model': mname, 'error': str(e), 'path': str(mpath)}))

print('Done. all_results keys:', list(all_results.keys()))

Using device: cuda
=== Evaluating dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo on /lp-dev/amelia/data/pavsig/training/new_test_manifest_mono.jsonl ===
[NeMo I 2026-05-31 11:26:14 mixins:184] Tokenizer SentencePieceTokenizer initialized with 16384 tokens


[NeMo W 2026-05-31 11:26:17 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    input_cfg: /lp-dev/amelia/inclusive-asr-moe/data/multilingual/train_cv.yaml
    skip_missing_manifest_entries: true
    num_workers: 8
    shuffle: true
    normalize_text: true
    use_lhotse: true
    force_iterable_dataset: true
    min_duration: 0.5
    max_duration: 40.0
    num_buckets: 10
    use_distributed_sampler: false
    bucket_duration_bins:
    - 1.0
    - 3.0
    - 5.0
    - 7.0
    - 9.0
    - 11.0
    - 13.0
    - 15.0
    - 20.0
    - 25.0
    bucket_batch_size:
    - 576
    - 432
    - 288
    - 216
    - 180
    - 144
    - 126
    - 108
    - 72
    - 54
    use_bucketing: true
    
[NeMo W 2026-05-31 11:26:17 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_valida

[NeMo I 2026-05-31 11:26:20 save_restore_connector:286] Model EncDecCTCModelBPE was successfully restored from /lp-dev/amelia/inclusive-asr-moe/experiments/NEW/multilingual/dense/dense_fastconformer_multilingual_2026-04-22_13-48-35/2026-04-22_13-48-44/checkpoints/dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo.


[NeMo W 2026-05-31 11:26:20 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-05-31 11:26:20 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-05-31 11:26:21 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-05-31 11:26:21 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

Saved results for dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo
=== Evaluating moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo on /lp-dev/amelia/data/pavsig/training/new_test_manifest_mono.jsonl ===
[NeMo I 2026-05-31 11:29:45 mixins:184] Tokenizer SentencePieceTokenizer initialized with 16384 tokens


...

Saved results for moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo
=== Evaluating moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo on /lp-dev/amelia/data/pavsig/training/new_test_manifest_mono.jsonl ===
[NeMo I 2026-05-31 11:37:34 mixins:184] Tokenizer SentencePieceTokenizer initialized with 16384 tokens


[NeMo W 2026-05-31 11:37:37 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    input_cfg: /lp-dev/amelia/inclusive-asr-moe/data/multilingual/train_child.yaml
    skip_missing_manifest_entries: true
    num_workers: 4
    shuffle: true
    normalize_text: true
    use_lhotse: true
    force_iterable_dataset: true
    min_duration: 1.0
    max_duration: 30.0
    num_buckets: 10
    use_distributed_sampler: false
    bucket_duration_bins:
    - 1.0
    - 3.0
    - 5.0
    - 7.0
    - 9.0
    - 11.0
    - 13.0
    - 15.0
    - 20.0
    - 25.0
    bucket_batch_size:
    - 576
    - 432
    - 288
    - 216
    - 180
    - 144
    - 126
    - 108
    - 72
    - 54
    use_bucketing: true
    
[NeMo W 2026-05-31 11:37:37 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_val

[NeMo I 2026-05-31 11:37:38 conformer_moe_encoder:467] Created ConformerMoEEncoder with 17 layers, 8 experts per layer, top-2 routing [global shared router]
[NeMo I 2026-05-31 11:37:41 save_restore_connector:286] Model EncDecCTCModelBPE was successfully restored from /lp-dev/amelia/inclusive-asr-moe/experiments/NEW/multilingual_child/moe/moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09/2026-05-09_10-14-25/checkpoints/moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo.


[NeMo W 2026-05-31 11:37:41 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-05-31 11:37:41 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-05-31 11:37:42 dataloader:879] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-05-31 11:37:42 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

Saved results for moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo
Done. all_results keys: ['dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo', 'moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo', 'dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo', 'moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo', 'moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo']


## Inspect results: per-model summaries and top error words

Run the cell below to print summaries and show top problematic words and speaker WERs for a chosen model.

In [158]:
# Per-word summary across models: accuracy table and plots
import matplotlib.pyplot as plt
import numpy as np

if 'all_results' not in globals() or not all_results:
    print('No evaluation results found in `all_results`. Run the evaluation cell first.')
else:
    rows = []
    words_set = set()
    for m, r in all_results.items():
        if isinstance(r, dict) and 'error' in r:
            continue
        utts = r['utterances']
        # build per-word confusion
        conf = defaultdict(Counter)
        totals = Counter()
        correct = Counter()
        for u in utts:
            ref = (u.get('ref') or '').split()[0] if u.get('ref') else ''
            hyp = (u.get('hyp') or '').split()[0] if u.get('hyp') else ''
            canonical = WORD_MAP.get(ref, ref)
            totals[canonical] += 1
            conf[canonical][hyp] += 1
            predicted_ok = False
            if hyp in CANONICAL_TO_KEYS.get(canonical, set()):
                predicted_ok = True
            else:
                if hyp in WORD_MAP and WORD_MAP[hyp] == canonical:
                    predicted_ok = True
            if predicted_ok:
                correct[canonical] += 1
            words_set.update([canonical])
        for w in totals:
            top_preds = conf[w].most_common(3)
            top_str = ';'.join([f"{p}:{c}" for p,c in top_preds])
            acc = correct.get(w,0)/totals[w] if totals[w]>0 else 0.0
            rows.append({'model': m, 'word': w, 'count': totals[w], 'correct': correct.get(w,0), 'accuracy': acc, 'top_preds': top_str})

    df_all = pd.DataFrame(rows)
    # Save full table
    out_all = Path('analysis_outputs') / 'word_accuracy_by_model.csv'
    df_all.to_csv(out_all, index=False)
    print('Saved per-word accuracy table to', out_all)

    # Pivot accuracy table: rows words, columns models
    pivot = df_all.pivot(index='word', columns='model', values='accuracy').fillna(0)
    pivot = pivot.reindex(sorted(pivot.index))
    out_pivot = Path('analysis_outputs') / 'word_accuracy_pivot.csv'
    pivot.to_csv(out_pivot)
    print('Saved pivot table to', out_pivot)

    # Plot per-model bar charts
    for m in pivot.columns:
        fig, ax = plt.subplots(figsize=(10,4))
        data = pivot[m].sort_values(ascending=False)
        data.plot(kind='bar', ax=ax)
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0,1)
        ax.set_title(f'Per-word accuracy — {m}')
        plt.tight_layout()
        out_png = Path('analysis_outputs') / f'{m}_word_accuracy.png'
        fig.savefig(out_png)
        plt.close(fig)
        print('Saved plot for', m, '->', out_png)

    # Summary: top 5 worst words per model
    print('\nTop 5 worst words per model (by accuracy):')
    for m in pivot.columns:
        worst = pivot[m].sort_values().head(5)
        print('\nModel:', m)
        for w, a in worst.items():
            print(f"  {w}: {a:.2f}")

    print('\nDone: per-word summary and plots saved to analysis_outputs/')


Saved per-word accuracy table to analysis_outputs/word_accuracy_by_model.csv
Saved pivot table to analysis_outputs/word_accuracy_pivot.csv
Saved plot for dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo -> analysis_outputs/dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo_word_accuracy.png
Saved plot for dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo -> analysis_outputs/dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo_word_accuracy.png
Saved plot for moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo -> analysis_outputs/moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo_word_accuracy.png
Saved plot for moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo -> analysis_outputs/moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo_word_accuracy.png
Saved plot for moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo -> analysis_outputs/moe_fastconformer_multilingual

In [164]:
# Most-mistaken words and their top confusions per model
from collections import defaultdict, Counter

if 'all_results' not in globals() or not all_results:
    print('No evaluation results found in `all_results`. Run the evaluation cell first.')
else:
    for m, r in all_results.items():
        print('\n===', m)
        if isinstance(r, dict) and 'error' in r:
            print('Model error:', r['error'])
            continue
        # Build canonical -> predicted canonical counts
        conf = defaultdict(Counter)
        totals = Counter()
        correct = Counter()
        for u in r['utterances']:
            ref = (u.get('ref') or '').split()[0] if u.get('ref') else ''
            hyp = (u.get('hyp') or '').split()[0] if u.get('hyp') else ''
            ref_can = WORD_MAP.get(ref, ref)
            hyp_can = WORD_MAP.get(hyp, hyp)
            totals[ref_can] += 1
            conf[ref_can][hyp_can] += 1
            if hyp_can == ref_can:
                correct[ref_can] += 1

        # Sort by error count desc
        err_rows = []
        for w in totals:
            errs = totals[w] - correct.get(w, 0)
            err_rows.append((w, totals[w], correct.get(w, 0), errs))
        err_rows.sort(key=lambda x: x[3], reverse=True)

        print('Most mistaken words (word | total | correct | errors):')
        for w, tot, cor, errs in err_rows[:15]:
            print(f"  {w:>15} | {tot:>3} | {cor:>3} | {errs:>3}")
            # list top 3 wrong predictions (exclude correct)
            top_preds = [(p, c) for p, c in conf[w].most_common(5) if p != w]
            if top_preds:
                print('    mistaken for:', ', '.join([f"{p}({c})" for p, c in top_preds[:3]]))

        # Save per-model confusion table
        out_csv = Path('analysis_outputs') / f"{m}_confusion_summary.csv"
        flat = []
        for w in totals:
            for p, c in conf[w].items():
                flat.append({'word': w, 'pred': p, 'count': c, 'total': totals[w]})
        pd.DataFrame(flat).to_csv(out_csv, index=False)
        print('Saved confusion summary to', out_csv)



=== dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo
Most mistaken words (word | total | correct | errors):
           szalik |  40 |   0 |  40
    mistaken for: shalik(14), shall(3), f(2)
             żaba |  40 |   0 |  40
    mistaken for: jaab(3), jaabba(3), aberber(2)
               ca |  40 |   0 |  40
    mistaken for: co(6), sam(5), t(5)
              cia |  40 |   0 |  40
    mistaken for: ja(6), chap(4), shall(3)
           dżokej |  40 |   0 |  40
    mistaken for: to(8), (6), j(3)
               ia |  40 |   0 |  40
    mistaken for: eat(17), e(12), a(4)
            radza |  40 |   0 |  40
    mistaken for: and(4), it(2), rada(2)
           żyrafa |  39 |   0 |  39
    mistaken for: że(3), sure(2), s(1)
             jeże |  38 |   0 |  38
    mistaken for: yes(12), jera(3), ja(2)
           czapka |  40 |   4 |  36
    mistaken for: chapka(2), czabka(2), tabka(2)
            zegar |  41 |   6 |  35
    mistaken for: zagad(3), zag(3), sei(2)
           siatka |  35

In [159]:
# Compare models by overall WER and error-type breakdown (S/D/I)
import matplotlib.pyplot as plt

if 'all_results' not in globals() or not all_results:
    print('No `all_results` available — run the evaluation cell first.')
else:
    rows = []
    # canonical word list (for confusion heatmaps)
    canonical_words = sorted(CANONICAL_TO_KEYS.keys())

    for m, r in all_results.items():
        if isinstance(r, dict) and 'error' in r:
            rows.append({'model': m, 'S': None, 'D': None, 'I': None, 'words': None, 'WER': None, 'error': r['error']})
            continue
        s = r['summary']['S']
        d = r['summary']['D']
        i = r['summary']['I']
        w = r['summary']['words']
        wer = r['summary'].get('WER', (s + d + i) / w if w else None)
        rows.append({'model': m, 'S': s, 'D': d, 'I': i, 'words': w, 'WER': wer})

    df_models = pd.DataFrame(rows).set_index('model')
    # Save CSV
    out_model_csv = Path('analysis_outputs') / 'model_wers.csv'
    df_models.to_csv(out_model_csv)
    print('Saved model WER summary to', out_model_csv)

    # Bar plot: WER per model
    fig, ax = plt.subplots(figsize=(8,4))
    plot_df = df_models.dropna(subset=['WER'])
    if not plot_df.empty:
        plot_df['WER'].sort_values().plot(kind='bar', ax=ax)
        ax.set_ylabel('WER (proportion)')
        ax.set_ylim(0, max(plot_df['WER'].max()*1.1, 0.01))
        ax.set_title('Model WER on PAVSig test')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        out_png = Path('analysis_outputs') / 'models_WER_comparison.png'
        fig.savefig(out_png)
        plt.close(fig)
        print('Saved WER comparison plot to', out_png)
    else:
        print('No WER values to plot.')

    # Stacked bar: S/D/I normalized by total words
    fig, ax = plt.subplots(figsize=(8,4))
    plot_df2 = df_models.fillna(0).loc[:, ['S','D','I']]
    if not plot_df2.empty:
        plot_df2_norm = plot_df2.div(plot_df2.sum(axis=1).replace(0,1), axis=0)
        plot_df2_norm[['S','D','I']].plot(kind='bar', stacked=True, ax=ax)
        ax.set_ylabel('Proportion of errors (S/D/I)')
        ax.set_title('Error type breakdown per model (normalized)')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Error type')
        plt.tight_layout()
        out_png2 = Path('analysis_outputs') / 'models_error_types.png'
        fig.savefig(out_png2)
        plt.close(fig)
        print('Saved error-type breakdown to', out_png2)
    else:
        print('No error-type data to plot.')

    # Optional: per-model confusion heatmaps (canonical -> predicted-canonical)
    try:
        import numpy as np
        for m, r in all_results.items():
            if isinstance(r, dict) and 'error' in r:
                continue
            # build confusion canonical->canonical
            matrix = pd.DataFrame(0, index=canonical_words, columns=canonical_words)
            for u in r['utterances']:
                ref_tok = (u.get('ref') or '').split()[0] if u.get('ref') else ''
                hyp_tok = (u.get('hyp') or '').split()[0] if u.get('hyp') else ''
                ref_can = WORD_MAP.get(ref_tok, ref_tok)
                # map hyp to canonical if possible
                hyp_can = WORD_MAP.get(hyp_tok, hyp_tok)
                if ref_can not in matrix.index:
                    # skip unknown refs
                    continue
                if hyp_can not in matrix.columns:
                    # add unknown hyp column
                    matrix[hyp_can] = 0
                matrix.at[ref_can, hyp_can] += 1
            # plot heatmap
            fig, ax = plt.subplots(figsize=(10,8))
            im = ax.imshow(matrix.values, aspect='auto', cmap='Blues')
            ax.set_xticks(np.arange(len(matrix.columns)))
            ax.set_yticks(np.arange(len(matrix.index)))
            ax.set_xticklabels(matrix.columns, rotation=90, fontsize=8)
            ax.set_yticklabels(matrix.index, fontsize=8)
            ax.set_title(f'Confusion (ref -> hyp canonical) — {m}')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.tight_layout()
            out_h = Path('analysis_outputs') / f'{m}_confusion_heatmap.png'
            fig.savefig(out_h)
            plt.close(fig)
            print('Saved confusion heatmap for', m, '->', out_h)
    except Exception as e:
        print('Skipping heatmaps (error):', e)

    print('\nModel summary:')
    display(df_models)


Saved model WER summary to analysis_outputs/model_wers.csv
Saved WER comparison plot to analysis_outputs/models_WER_comparison.png
Saved error-type breakdown to analysis_outputs/models_error_types.png


[NeMo W 2026-05-31 11:39:29 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:29 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:29 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Saved confusion heatmap for dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo -> analysis_outputs/dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo_confusion_heatmap.png


[NeMo W 2026-05-31 11:39:31 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:31 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:31 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Saved confusion heatmap for moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo -> analysis_outputs/moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo_confusion_heatmap.png


[NeMo W 2026-05-31 11:39:33 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:33 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:33 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Saved confusion heatmap for dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo -> analysis_outputs/dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo_confusion_heatmap.png


[NeMo W 2026-05-31 11:39:34 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:34 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
      matrix[hyp_can] = 0
    
[NeMo W 2026-05-31 11:39:34 nemo_logging:364] /tmp/ipykernel_1585778/2624540967.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Saved confusion heatmap for moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo -> analysis_outputs/moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo_confusion_heatmap.png
Saved confusion heatmap for moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo -> analysis_outputs/moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo_confusion_heatmap.png

Model summary:


,S,D,I,words,WER
model,,,,,
dense_fastconformer_multilingual_2026-04-22_13-48-35.nemo,470,12,153,506,1.254941
moe_fastconformer_multilingual_2026-04-22_13-48-13.nemo,494,4,217,506,1.413043
dense_fastconformer_child_multilingual_2026-05-09_09-11-55.nemo,437,0,38,506,0.938735
moe_fastconformer_child_multilingual_2026-05-07_21-03-06.nemo,459,14,19,506,0.972332
moe_fastconformer_child_multilingual_load_balancing_on_2026-05-09_10-14-09.nemo,474,6,28,506,1.003953


## Export and plotting helpers

Small helpers to save results and plot per-speaker WER distributions.

In [160]:
def save_results(name, results, out_dir=Path('analysis_outputs')):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    # If results contains an error (model failed), save a small error JSON and return
    if isinstance(results, dict) and 'error' in results:
        with open(out_dir / f'{name}_error.json', 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print('Saved error result to', out_dir / f'{name}_error.json')
        return
    with open(out_dir / f'{name}_summary.json', 'w', encoding='utf-8') as f:
        json.dump(results.get('summary', {}), f, indent=2, ensure_ascii=False)
    pd.DataFrame(results.get('utterances', [])).to_csv(out_dir / f'{name}_utterances.csv', index=False)
    results['per_speaker'].to_csv(out_dir / f'{name}_per_speaker.csv', index=False)
    print('Saved results to', out_dir)

def plot_speaker_wer(results, top_n=50):
    df = results['per_speaker']
    df2 = df.sort_values('WER', ascending=False).head(top_n)
    plt.figure(figsize=(10,6))
    plt.barh(df2['speaker'].astype(str), df2['WER']*100)
    plt.xlabel('WER (%)')
    plt.title('Top speakers by WER')
    plt.gca().invert_yaxis()
    plt.show()

In [161]:
# Use the specific working models (absolute paths provided by user)
models_to_run = [
    Path('/lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_adult_moe.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_child_moe_lb_off.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_child_moe_lb_on.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_fastconformer.nemo'),
    Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_fastconformer.nemo'),
]
models_to_run = [p for p in models_to_run if p.exists()]
print('Pinned models_to_run (existing):')
for p in models_to_run:
    print(' -', p)
if not models_to_run:
    print('Warning: none of the specified model files exist. Check paths or run listing cell.')

Pinned models_to_run (existing):
 - /lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_adult_moe.nemo
 - /lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_child_moe_lb_off.nemo
 - /lp-dev/amelia/inclusive-asr-moe/final_weights/multilingual_child_moe_lb_on.nemo
 - /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_fastconformer.nemo
 - /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_fastconformer.nemo


In [162]:
# WORD_MAP: map manifest-safe tokens to canonical orthographic forms
WORD_MAP = {
    "bocian": "bocian",
    "biegacz": "biegacz",
    "dzokej": "dżokej",
    "jeze": "jeże",
    "kaczka": "kaczka",
    "pies": "pies",
    "roza": "róża",
    "waz": "wąż",
    "zaba": "żaba",
    "zyrafa": "żyrafa",

    "bazie": "bazie",
    "cebula": "cebula",
    "czapka": "czapka",
    "dziadek": "dziadek",
    "dzwonek": "dzwonek",
    "kalosze": "kalosze",
    "koszyk": "koszyk",
    "ksiazka": "książka",
    "kucharz": "kucharz",
    "las": "las",
    "lekarz": "lekarz",
    "mazaki": "mazaki",
    "noz": "nóż",
    "owoce": "owoce",
    "parasol": "parasol",
    "rzeka": "rzeka",
    "salata": "sałata",
    "samolot": "samolot",
    "szafa": "szafa",
    "szalik": "szalik",
    "sznurek": "sznurek",
    "szufelka": "szufelka",
    "warzywa": "warzywa",
    "widelec": "widelec",
    "zabawki": "zabawki",
    "zegar": "zegar",
    "kuchnia": "kuchnia",
    "straż": "straż",
    "strazak": "strażak",
    "ciastka": "ciastka",
    "ciasto": "ciasto",
    "rybka": "rybka",
    "ryba": "ryba",
    "sanki": "sanki",
    "lyzwy": "łyżwy",
    "siatka": "siatka",
    "siatkówka": "siatkówka",
    "jeż": "jeż",
    "żaba": "żaba",
    "stawek": "stawek",
    "staw": "staw",
    "zaba2": "żaba",

    "ca": "ca","cia": "cia","cza": "cza","drza": "drza","dza": "dza","dzia": "dzia",
    "dża": "dża","sa": "sa","sia": "sia","sza": "sza","za": "za","zia": "zia",
    "ża": "ża","aca": "aca","acia": "acia","acza": "acza","asa": "asa","asia": "asia",
    "asza": "asza","aza": "aza","azia": "azia",

    "kasza": "kasza","koza": "koza","lodzie": "łodzie","lokiec": "łokieć",
    "pajac": "pajac","paz": "pąż","sadzawka": "sadzawka","taca": "taca",
    "wpasie": "w pasie","zarowka": "żarówka","ziarno": "ziarno",
    "ia": "ia","iu": "iu","radza": "radza","rza": "rza",
}

# Build inverse mapping: canonical -> set of manifest tokens (keys and the canonical itself)
CANONICAL_TO_KEYS = {}
for k, v in WORD_MAP.items():
    CANONICAL_TO_KEYS.setdefault(v, set()).add(k)
    CANONICAL_TO_KEYS.setdefault(v, set()).add(v)

print('WORD_MAP loaded, canonical entries:', len(CANONICAL_TO_KEYS))

WORD_MAP loaded, canonical entries: 86
